# Coffee Standard v8 — leakage-safe external benchmark
Membentuk satu benchmark eksternal dari satu gambar representatif per identitas induk dan 18 kelas yang ekuivalen langsung dengan SNI-21. Tidak melakukan training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
import hashlib, importlib, json, os, shutil, subprocess, sys, tarfile
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/af2-igem-paired-confirmation'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root,require_project_artifact
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=('bundles/coffee-detection-with-standard-v8-yolov8.tar','evidence/coffee-detection-with-standard-v8/dataset_audit.json'))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/coffee-detection-with-standard-v8-yolov8.tar')
RAW=Path('/content/coffee-standard-v8-raw'); OUTPUT=Path('/content/coffee-standard-v8-external')
for path in (RAW,OUTPUT):
    if path.exists(): shutil.rmtree(path)
RAW.mkdir();
with tarfile.open(ARCHIVE,'r') as archive: archive.extractall(RAW,filter='data')
print('PROJECT:',PROJECT_ROOT); print('SOURCE:',ARCHIVE)


In [ ]:
from coffee_detector.data.prepare_coffee_standard_external import prepare_external
summary=prepare_external(RAW,OUTPUT)
print(json.dumps(summary,indent=2,ensure_ascii=False))


In [ ]:
BUNDLE=PROJECT_ROOT/'bundles/coffee-detection-with-standard-v8-external-sni18-v1.tar'
EVIDENCE=PROJECT_ROOT/'evidence/coffee-detection-with-standard-v8/external_benchmark_summary.json'
with tarfile.open(BUNDLE,'w') as archive: archive.add(OUTPUT,arcname=OUTPUT.name)
shutil.copy2(OUTPUT/'external_summary.json',EVIDENCE)
digest=hashlib.sha256()
with BUNDLE.open('rb') as handle:
    for chunk in iter(lambda:handle.read(8*1024*1024),b''): digest.update(chunk)
print('BUNDLE:',BUNDLE); print('SHA256:',digest.hexdigest()); print('SUMMARY:',EVIDENCE)
print('TRAINING AUTHORIZED:',summary['training_authorized'])
print('Kirim summary dan tabel kelas sebelum evaluasi checkpoint.')


In [ ]:
import pandas as pd
from IPython.display import display
display(pd.DataFrame(summary['class_distribution']))
